In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

RAW = Path("../data/raw")
CLEANED = Path("../data/cleaned")
CLEANED.mkdir(parents=True, exist_ok=True)

print("Libraries loaded.")
print(f"Raw path exists: {RAW.exists()}")
print(f"Cleaned path exists: {CLEANED.exists()}")

Libraries loaded.
Raw path exists: True
Cleaned path exists: True


LOAD ALL 9 RAW FILES

In [3]:
orders = pd.read_csv(RAW / "olist_orders_dataset.csv")
order_items = pd.read_csv(RAW / "olist_order_items_dataset.csv")
payments = pd.read_csv(RAW / "olist_order_payments_dataset.csv")
reviews = pd.read_csv(RAW / "olist_order_reviews_dataset.csv")
customers = pd.read_csv(RAW / "olist_customers_dataset.csv")
sellers = pd.read_csv(RAW / "olist_sellers_dataset.csv")
products = pd.read_csv(RAW / "olist_products_dataset.csv")
geolocation = pd.read_csv(RAW / "olist_geolocation_dataset.csv")
translation = pd.read_csv(RAW / "product_category_name_translation.csv")

print("All 9 files loaded.")
for name, df in [("orders", orders), ("order_items", order_items),
                 ("payments", payments), ("reviews", reviews),
                 ("customers", customers), ("sellers", sellers),
                 ("products", products), ("geolocation", geolocation),
                 ("translation", translation)]:
    print(f"  {name}: {df.shape}")

All 9 files loaded.
  orders: (99441, 8)
  order_items: (112650, 7)
  payments: (103886, 5)
  reviews: (99224, 7)
  customers: (99441, 5)
  sellers: (3095, 4)
  products: (32951, 9)
  geolocation: (1000163, 5)
  translation: (71, 2)


Cast all datetime columns

In [4]:
# --- ORDERS ---
dt_cols_orders = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

print("BEFORE orders dtypes:")
print(orders[dt_cols_orders].dtypes)
print("Nulls before:")
print(orders[dt_cols_orders].isnull().sum())

for col in dt_cols_orders:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

print("\nAFTER orders dtypes:")
print(orders[dt_cols_orders].dtypes)
print("Nulls after (should be same or slightly higher if any unparseable strings existed):")
print(orders[dt_cols_orders].isnull().sum())

BEFORE orders dtypes:
order_purchase_timestamp         object
order_approved_at                object
order_delivered_carrier_date     object
order_delivered_customer_date    object
order_estimated_delivery_date    object
dtype: object
Nulls before:
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

AFTER orders dtypes:
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object
Nulls after (should be same or slightly higher if any unparseable strings existed):
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date      

In [5]:
# --- ORDER_ITEMS ---
print("BEFORE order_items shipping_limit_date dtype:", order_items["shipping_limit_date"].dtype)
order_items["shipping_limit_date"] = pd.to_datetime(order_items["shipping_limit_date"], errors="coerce")
print("AFTER order_items shipping_limit_date dtype:", order_items["shipping_limit_date"].dtype)
print("Nulls introduced:", order_items["shipping_limit_date"].isnull().sum())

BEFORE order_items shipping_limit_date dtype: object
AFTER order_items shipping_limit_date dtype: datetime64[ns]
Nulls introduced: 0


In [6]:
# --- REVIEWS ---
dt_cols_reviews = ["review_creation_date", "review_answer_timestamp"]

print("BEFORE reviews dtypes:")
print(reviews[dt_cols_reviews].dtypes)

for col in dt_cols_reviews:
    reviews[col] = pd.to_datetime(reviews[col], errors="coerce")

print("\nAFTER reviews dtypes:")
print(reviews[dt_cols_reviews].dtypes)
print("Nulls after:")
print(reviews[dt_cols_reviews].isnull().sum())

BEFORE reviews dtypes:
review_creation_date       object
review_answer_timestamp    object
dtype: object

AFTER reviews dtypes:
review_creation_date       datetime64[ns]
review_answer_timestamp    datetime64[ns]
dtype: object
Nulls after:
review_creation_date       0
review_answer_timestamp    0
dtype: int64


Geolocation - deduplicate, aggregate, clip outliers

In [7]:
# --- BEFORE ---
print("BEFORE geolocation shape:", geolocation.shape)
print("Exact duplicate rows:", geolocation.duplicated().sum())
print("Lat range: min =", geolocation["geolocation_lat"].min(), "| max =", geolocation["geolocation_lat"].max())
print("Lon range: min =", geolocation["geolocation_lng"].min(), "| max =", geolocation["geolocation_lng"].max())

# Step 3a: Drop exact duplicates first
geolocation = geolocation.drop_duplicates()
print("\nAfter dropping exact duplicates:", geolocation.shape)

# Step 3b: Clip to Brazil bounding box
# Verified bounds: lat -33.75 to 5.27, lon -73.99 to -28.85
BRAZIL_LAT_MIN, BRAZIL_LAT_MAX = -33.75, 5.27
BRAZIL_LON_MIN, BRAZIL_LON_MAX = -73.99, -28.85

outliers = geolocation[
    (geolocation["geolocation_lat"] < BRAZIL_LAT_MIN) |
    (geolocation["geolocation_lat"] > BRAZIL_LAT_MAX) |
    (geolocation["geolocation_lng"] < BRAZIL_LON_MIN) |
    (geolocation["geolocation_lng"] > BRAZIL_LON_MAX)
]
print("\nOutlier rows outside Brazil bounding box:", len(outliers))
print(outliers[["geolocation_zip_code_prefix", "geolocation_lat", "geolocation_lng", "geolocation_city", "geolocation_state"]].to_string())

geolocation = geolocation[
    (geolocation["geolocation_lat"] >= BRAZIL_LAT_MIN) &
    (geolocation["geolocation_lat"] <= BRAZIL_LAT_MAX) &
    (geolocation["geolocation_lng"] >= BRAZIL_LON_MIN) &
    (geolocation["geolocation_lng"] <= BRAZIL_LON_MAX)
]
print("\nAfter clipping outliers:", geolocation.shape)

# Step 3c: Aggregate to one lat/lon per zip prefix (median is more outlier-resistant than mean)
geo_agg = (
    geolocation
    .groupby("geolocation_zip_code_prefix")
    .agg(
        geolocation_lat=("geolocation_lat", "median"),
        geolocation_lng=("geolocation_lng", "median"),
        geolocation_city=("geolocation_city", lambda x: x.mode()[0]),
        geolocation_state=("geolocation_state", lambda x: x.mode()[0])
    )
    .reset_index()
)

print("\nAFTER aggregation — geo_agg shape:", geo_agg.shape)
print("Lat range: min =", geo_agg["geolocation_lat"].min(), "| max =", geo_agg["geolocation_lat"].max())
print("Lon range: min =", geo_agg["geolocation_lng"].min(), "| max =", geo_agg["geolocation_lng"].max())
print(geo_agg.head())

BEFORE geolocation shape: (1000163, 5)


Exact duplicate rows: 261831
Lat range: min = -36.6053744107061 | max = 45.06593318269697
Lon range: min = -101.46676644931476 | max = 121.10539381057764

After dropping exact duplicates: (738332, 5)

Outlier rows outside Brazil bounding box: 27
        geolocation_zip_code_prefix  geolocation_lat  geolocation_lng         geolocation_city geolocation_state
387565                        18243        28.008978       -15.536867  bom retiro da esperanca                SP
513631                        28165        41.614052        -8.411675      vila nova de campos                RJ
513643                        28155       -34.586422       -58.732101              santa maria                RJ
513754                        28155        42.439286        13.820214              santa maria                RJ
514429                        28333        38.381672        -6.328200                   raposo                RJ
516682                        28595        43.684961        -7.411080       

Deduplicate reviews on review_id

In [8]:
print("BEFORE reviews shape:", reviews.shape)
print("Non-unique review_ids:", reviews["review_id"].duplicated().sum())
print("Sample duplicate:")
dup_id = reviews[reviews["review_id"].duplicated(keep=False)]["review_id"].iloc[0]
print(reviews[reviews["review_id"] == dup_id][
    ["review_id", "order_id", "review_score", "review_creation_date"]
].to_string())

reviews_clean = reviews.drop_duplicates(subset="review_id", keep="first")

print("\nAFTER reviews shape:", reviews_clean.shape)
print("Non-unique review_ids remaining:", reviews_clean["review_id"].duplicated().sum())
print(f"Rows dropped: {len(reviews) - len(reviews_clean)}")

BEFORE reviews shape: (99224, 7)
Non-unique review_ids: 814
Sample duplicate:
                              review_id                          order_id  review_score review_creation_date
200    28642ce6250b94cc72bc85960aec6c62  e239d280236cdd3c40cb2c033f681d1c             5           2018-03-25
58385  28642ce6250b94cc72bc85960aec6c62  bc42a955f289870d5789e6e437206300             5           2018-03-25

AFTER reviews shape: (98410, 7)
Non-unique review_ids remaining: 0
Rows dropped: 814


Aggregate payments by order_id

In [9]:
print("BEFORE payments shape:", payments.shape)
print("payment_type distribution:")
print(payments["payment_type"].value_counts())
print("\n'not_defined' rows:")
print(payments[payments["payment_type"] == "not_defined"])

# Flag and remove the 3 not_defined rows — they have no analytical value
payments_flagged = payments[payments["payment_type"] == "not_defined"].copy()
payments_clean_raw = payments[payments["payment_type"] != "not_defined"].copy()
print(f"\nRows removed (not_defined): {len(payments_flagged)}")

# Aggregate to one row per order
payments_agg = (
    payments_clean_raw
    .groupby("order_id")
    .agg(
        payment_value=("payment_value", "sum"),
        payment_installments=("payment_installments", "max"),
        payment_type_primary=("payment_type", lambda x: x.value_counts().index[0]),
        payment_type_count=("payment_type", "nunique")
    )
    .reset_index()
)

print("\nAFTER payments_agg shape:", payments_agg.shape)
print("payment_value nulls:", payments_agg["payment_value"].isnull().sum())
print("Sample:")
print(payments_agg.head())
print("\nOrders with split payment types:", (payments_agg["payment_type_count"] > 1).sum())

BEFORE payments shape: (103886, 5)
payment_type distribution:
payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

'not_defined' rows:
                               order_id  payment_sequential payment_type  \
51280  4637ca194b6387e2d538dc89b124b0ee                   1  not_defined   
57411  00b1cb0320190ca0daa2c88b35206009                   1  not_defined   
94427  c8c528189310eaa44a745b8d9d26908b                   1  not_defined   

       payment_installments  payment_value  
51280                     1            0.0  
57411                     1            0.0  
94427                     1            0.0  

Rows removed (not_defined): 3

AFTER payments_agg shape: (99437, 5)
payment_value nulls: 0
Sample:
                           order_id  payment_value  payment_installments  \
0  00010242fe8c5a6d1ba2dd792cb16214          72.19                     2   
1  00018f77f2f0320c557190d7a144bdd3 

Products - join translation, handle 13 unmapped categories, impute dimension nulls

In [10]:
print("BEFORE products shape:", products.shape)
print("Category nulls:", products["product_category_name"].isnull().sum())
print("Dimension nulls:")
dim_cols = ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]
print(products[dim_cols].isnull().sum())

# Step 6a: Join translation
products_clean = products.merge(
    translation,
    on="product_category_name",
    how="left"
)

print("\nAfter joining translation:")
print("Rows with no English translation:", products_clean["product_category_name_english"].isnull().sum())
print("Categories with no translation:")
unmapped = products_clean[products_clean["product_category_name_english"].isnull()]["product_category_name"].unique()
print(unmapped)

BEFORE products shape: (32951, 9)
Category nulls: 610
Dimension nulls:
product_weight_g     2
product_length_cm    2
product_height_cm    2
product_width_cm     2
dtype: int64

After joining translation:
Rows with no English translation: 623
Categories with no translation:
[nan 'pc_gamer' 'portateis_cozinha_e_preparadores_de_alimentos']


In [11]:
# Step 6b: Handle unmapped categories
# Option: transliterate manually for the 13 known unmapped ones
# This is the honest approach — don't just call them "unknown"

manual_translations = {
    "portateis_cozinha_e_preparadores_de_alimentos": "portable_kitchen_food_prep",
    "pc_gamer": "gaming_pc",
    "fashion_roupa_infanto_juvenil": "fashion_kids_clothing",
    # Add the remaining ones you found in Day 1 profiling here
    # Print unmapped above to see all 13 and fill this dict out
}

products_clean["product_category_name_english"] = products_clean.apply(
    lambda row: manual_translations.get(row["product_category_name"], row["product_category_name_english"])
    if pd.isnull(row["product_category_name_english"]) else row["product_category_name_english"],
    axis=1
)

# Fill remaining nulls (products with no category at all)
products_clean["product_category_name_english"] = products_clean["product_category_name_english"].fillna("uncategorized")

print("Unmapped after manual fix:", products_clean["product_category_name_english"].isnull().sum())
print("'uncategorized' count:", (products_clean["product_category_name_english"] == "uncategorized").sum())

Unmapped after manual fix: 0
'uncategorized' count: 610


In [12]:
# Step 6c: Impute dimension nulls with category median
# Business justification: a product's dimensions are best estimated from similar products
# in the same category. Global median would be less accurate.

print("BEFORE dimension nulls:")
print(products_clean[dim_cols].isnull().sum())

for col in dim_cols:
    category_medians = products_clean.groupby("product_category_name_english")[col].transform("median")
    global_median = products_clean[col].median()
    products_clean[col] = products_clean[col].fillna(category_medians).fillna(global_median)

print("\nAFTER dimension nulls:")
print(products_clean[dim_cols].isnull().sum())
print("Any nulls remaining:", products_clean[dim_cols].isnull().any().any())

BEFORE dimension nulls:
product_weight_g     2
product_length_cm    2
product_height_cm    2
product_width_cm     2
dtype: int64

AFTER dimension nulls:
product_weight_g     0
product_length_cm    0
product_height_cm    0
product_width_cm     0
dtype: int64
Any nulls remaining: False


Orders - flag ghost orders, build derived columns

In [13]:
print("BEFORE orders shape:", orders.shape)

# Step 7a: Flag the 775 ghost orders
orders_with_items = set(order_items["order_id"].unique())
orders["has_items"] = orders["order_id"].isin(orders_with_items)
ghost_orders = orders[~orders["has_items"]]
print(f"\nGhost orders flagged: {len(ghost_orders)}")
print("Ghost order status distribution:")
print(ghost_orders["order_status"].value_counts())

# Step 7b: Flag the 1 order with no payment
orders_with_payment = set(payments_agg["order_id"].unique())
orders["has_payment"] = orders["order_id"].isin(orders_with_payment)
print(f"\nOrders without payment record: {(~orders['has_payment']).sum()}")

# Step 7c: Derived columns
# days_to_deliver: actual calendar days from purchase to customer receipt
orders["days_to_deliver"] = (
    orders["order_delivered_customer_date"] - orders["order_purchase_timestamp"]
).dt.days

# days_to_estimated: how far off the estimate was (negative = early, positive = late)
orders["days_vs_estimate"] = (
    orders["order_delivered_customer_date"] - orders["order_estimated_delivery_date"]
).dt.days

# is_late: delivered after the estimated date
orders["is_late"] = orders["days_vs_estimate"] > 0

# approval_lag: time from purchase to payment approval in hours
orders["approval_lag_hours"] = (
    orders["order_approved_at"] - orders["order_purchase_timestamp"]
).dt.total_seconds() / 3600

print("\nAFTER derived columns:")
print(orders[["days_to_deliver", "days_vs_estimate", "is_late", "approval_lag_hours"]].describe())
print(f"\nLate deliveries: {orders['is_late'].sum()} out of {orders['is_late'].notna().sum()} delivered orders")
print(f"Late delivery rate: {orders['is_late'].mean():.1%}")

BEFORE orders shape: (99441, 8)

Ghost orders flagged: 775
Ghost order status distribution:
order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1
Name: count, dtype: int64

Orders without payment record: 4

AFTER derived columns:
       days_to_deliver  days_vs_estimate  approval_lag_hours
count     96476.000000      96476.000000        99281.000000
mean         12.094086        -11.876881           10.419094
std           9.551746         10.183854           26.038004
min           0.000000       -147.000000            0.000000
25%           6.000000        -17.000000            0.215000
50%          10.000000        -12.000000            0.343333
75%          15.000000         -7.000000           14.580833
max         209.000000        188.000000         4509.180556

Late deliveries: 6535 out of 99441 delivered orders
Late delivery rate: 6.6%


In [14]:
# Step 7d: Sanity checks on derived columns
# Negative days_to_deliver would mean delivered before purchase — a data error
negative_delivery = orders[orders["days_to_deliver"] < 0]
print(f"Orders with negative days_to_deliver: {len(negative_delivery)}")
if len(negative_delivery) > 0:
    print(negative_delivery[["order_id", "order_purchase_timestamp",
                              "order_delivered_customer_date", "days_to_deliver"]])
    # Flag these — do not drop silently
    orders.loc[orders["days_to_deliver"] < 0, "days_to_deliver"] = np.nan
    print("Flagged as NaN. Count:", orders["days_to_deliver"].isnull().sum())

# Extreme delivery times — anything over 180 days is suspicious
extreme_delivery = orders[orders["days_to_deliver"] > 180]
print(f"\nOrders with delivery > 180 days: {len(extreme_delivery)}")
if len(extreme_delivery) > 0:
    print(extreme_delivery[["order_id", "days_to_deliver", "order_status"]])

Orders with negative days_to_deliver: 0

Orders with delivery > 180 days: 14
                               order_id  days_to_deliver order_status
11399  47b40429ed8cce3aee9199792275433f            191.0    delivered
19590  ca07593549f1816d26a572e06dc1eab6            209.0    delivered
31228  dfe5f68118c2576143240b8d78e5940a            186.0    delivered
38509  0f4519c5f1c541ddec9f21b3bddd533a            194.0    delivered
54480  2d7561026d542c8dbd8f0daeadf67a43            188.0    delivered
55619  1b3190b2dfa9d789e1f14c05b647a14a            208.0    delivered
61610  440d0d17af552815d15a9e41abe49359            195.0    delivered
62286  437222e3fd1b07396f1d9ba8c15fba59            187.0    delivered
66407  2ba1366baecad3c3536f27546d129017            181.0    delivered
68769  c27815f7e3dd0b926b58552628481575            187.0    delivered
70307  2fb597c2f772eca01b1f5c561bf6cc7b            194.0    delivered
74117  6e82dcfb5eada6283dba34f164e636f5            182.0    delivered
81401  2fe324

In [15]:
extreme_ids = orders[orders["days_to_deliver"] > 180]["order_id"].tolist()

extreme_detail = orders[orders["order_id"].isin(extreme_ids)][[
    "order_id",
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "days_to_deliver",
    "days_vs_estimate",
    "is_late"
]].copy()

print(extreme_detail.to_string())

                               order_id order_purchase_timestamp   order_approved_at order_delivered_carrier_date order_delivered_customer_date order_estimated_delivery_date  days_to_deliver  days_vs_estimate  is_late
11399  47b40429ed8cce3aee9199792275433f      2018-01-03 09:44:01 2018-01-03 10:31:15          2018-02-06 01:48:28           2018-07-13 20:51:31                    2018-01-19            191.0             175.0     True
19590  ca07593549f1816d26a572e06dc1eab6      2017-02-21 23:31:27 2017-02-23 02:35:15          2017-03-08 13:47:46           2017-09-19 14:36:39                    2017-03-22            209.0             181.0     True
31228  dfe5f68118c2576143240b8d78e5940a      2017-03-17 12:32:22 2017-03-17 12:32:22          2017-03-21 18:28:04           2017-09-19 18:13:19                    2017-04-19            186.0             153.0     True
38509  0f4519c5f1c541ddec9f21b3bddd533a      2017-03-09 13:26:57 2017-03-09 13:26:57          2017-03-22 05:28:14           2017

In [16]:
# Check how many delivered orders have Sept 19 as customer delivery date
sept19_2017 = orders[
    (orders["order_delivered_customer_date"].dt.month == 9) &
    (orders["order_delivered_customer_date"].dt.day == 19) &
    (orders["order_delivered_customer_date"].dt.year == 2017) &
    (orders["order_status"] == "delivered")
]

sept19_2018 = orders[
    (orders["order_delivered_customer_date"].dt.month == 9) &
    (orders["order_delivered_customer_date"].dt.day == 19) &
    (orders["order_delivered_customer_date"].dt.year == 2018) &
    (orders["order_status"] == "delivered")
]

print(f"Delivered orders with customer date = 2017-09-19: {len(sept19_2017)}")
print(f"Delivered orders with customer date = 2018-09-19: {len(sept19_2018)}")

print("\n2017-09-19 purchase date range:")
print(sept19_2017["order_purchase_timestamp"].describe())

print("\n2018-09-19 purchase date range:")
print(sept19_2018["order_purchase_timestamp"].describe())

Delivered orders with customer date = 2017-09-19: 282
Delivered orders with customer date = 2018-09-19: 3

2017-09-19 purchase date range:
count                              282
mean     2017-08-23 01:30:38.847517696
min                2017-02-21 23:31:27
25%      2017-08-29 04:17:34.750000128
50%                2017-09-07 20:35:56
75%                2017-09-11 21:58:39
max                2017-09-18 15:07:49
Name: order_purchase_timestamp, dtype: object

2018-09-19 purchase date range:
count                                3
mean     2018-06-01 17:14:36.666666752
min                2018-02-23 14:57:35
25%         2018-04-29 18:35:41.500000
50%                2018-07-03 22:13:48
75%         2018-07-20 18:23:07.500000
max                2018-08-06 14:32:27
Name: order_purchase_timestamp, dtype: object


In [17]:
# Flag all 285 orders with placeholder delivery dates
placeholder_mask = (
    (orders["order_delivered_customer_date"].dt.month == 9) &
    (orders["order_delivered_customer_date"].dt.day == 19) &
    (orders["order_delivered_customer_date"].dt.year.isin([2017, 2018])) &
    (orders["order_status"] == "delivered")
)

orders["has_placeholder_delivery_date"] = placeholder_mask

print(f"Orders flagged with placeholder delivery date: {placeholder_mask.sum()}")

# Null out days_to_deliver, days_vs_estimate, and is_late for these orders
# Their delivery timing data is unreliable and must not enter SLA calculations
orders.loc[placeholder_mask, "days_to_deliver"] = np.nan
orders.loc[placeholder_mask, "days_vs_estimate"] = np.nan
orders.loc[placeholder_mask, "is_late"] = np.nan

print("\nAfter nulling derived columns for placeholder orders:")
print(f"days_to_deliver nulls: {orders['days_to_deliver'].isnull().sum()}")
print(f"days_vs_estimate nulls: {orders['days_vs_estimate'].isnull().sum()}")
print(f"is_late nulls: {orders['is_late'].isnull().sum()}")

# Recheck late delivery rate on clean data only
clean_delivered = orders[
    (orders["order_status"] == "delivered") &
    (orders["has_placeholder_delivery_date"] == False) &
    (orders["is_late"].notna())
]

print(f"\nDelivered orders with reliable timing data: {len(clean_delivered)}")
print(f"Late deliveries: {clean_delivered['is_late'].sum()}")
print(f"Late delivery rate (clean): {clean_delivered['is_late'].mean():.1%}")
print(f"\ndays_to_deliver on clean orders:")
print(clean_delivered["days_to_deliver"].describe())

Orders flagged with placeholder delivery date: 285

After nulling derived columns for placeholder orders:
days_to_deliver nulls: 3250
days_vs_estimate nulls: 3250
is_late nulls: 285

Delivered orders with reliable timing data: 96193
Late deliveries: 6478
Late delivery rate (clean): 6.7%

days_to_deliver on clean orders:
count    96185.000000
mean        12.046265
std          9.201075
min          0.000000
25%          6.000000
50%         10.000000
75%         15.000000
max        191.000000
Name: days_to_deliver, dtype: float64


In [18]:
# Confirm the days_to_deliver null breakdown makes sense
print("days_to_deliver null breakdown:")
print(f"  Placeholder delivery date (285 flagged): {orders['has_placeholder_delivery_date'].sum()}")
print(f"  No customer delivery date at all: {orders['order_delivered_customer_date'].isnull().sum()}")
print(f"  Total days_to_deliver nulls: {orders['days_to_deliver'].isnull().sum()}")
print(f"  285 + 2965 = {285 + 2965}")

# One more check: min days_to_deliver is 0.0
# That means at least one order was purchased and delivered same day
# Verify this is real and not another artifact
same_day = orders[orders["days_to_deliver"] == 0]
print(f"\nSame-day deliveries (0 days): {len(same_day)}")
print(same_day[["order_id", "order_purchase_timestamp",
                "order_delivered_customer_date",
                "has_placeholder_delivery_date"]].head(5).to_string())

days_to_deliver null breakdown:
  Placeholder delivery date (285 flagged): 285
  No customer delivery date at all: 2965
  Total days_to_deliver nulls: 3250
  285 + 2965 = 3250

Same-day deliveries (0 days): 13
                               order_id order_purchase_timestamp order_delivered_customer_date  has_placeholder_delivery_date
395    38c1e3d4ed6a13cd0cf612d4c09766e9      2018-02-02 15:26:38           2018-02-03 15:05:56                          False
735    d3ca7b82c922817b06e5ca21165c5ea2      2017-11-16 13:54:08           2017-11-17 13:49:40                          False
31522  1d893dd7ca5f77ebf5f59f0d2017eee0      2017-06-19 08:19:45           2017-06-19 21:07:52                          False
37753  21a8ffca665bc7a1087d31751a7b7cbc      2017-05-31 12:00:35           2017-06-01 10:28:24                          False
38792  f3c6775ba3d2d9fe2826f93b71f12008      2017-07-04 11:37:47           2017-07-05 08:09:26                          False


Add has_comment to reviews

In [19]:
reviews_clean["has_comment"] = reviews_clean["review_comment_message"].notna()
print("has_comment distribution:")
print(reviews_clean["has_comment"].value_counts())
print(f"Comment rate: {reviews_clean['has_comment'].mean():.1%}")

has_comment distribution:
has_comment
False    57742
True     40668
Name: count, dtype: int64
Comment rate: 41.3%


Export all cleaned files

In [20]:
orders.to_csv(CLEANED / "orders_clean.csv", index=False)
print("orders_clean.csv exported")
print("Shape:", orders.shape)
print("Columns:", orders.columns.tolist())

orders_clean.csv exported
Shape: (99441, 15)
Columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'has_items', 'has_payment', 'days_to_deliver', 'days_vs_estimate', 'is_late', 'approval_lag_hours', 'has_placeholder_delivery_date']


In [21]:
from pathlib import Path
CLEANED = Path("../data/cleaned")
files = sorted(CLEANED.iterdir())
print(f"Files in data/cleaned: {len(files)}")
for f in files:
    df_check = __import__('pandas').read_csv(f)
    print(f"  {f.name}: {df_check.shape}")

Files in data/cleaned: 1
  orders_clean.csv: (99441, 15)


In [22]:
for name, var in [("orders", "orders"), 
                  ("order_items", "order_items"),
                  ("payments_agg", "payments_agg"), 
                  ("reviews_clean", "reviews_clean"),
                  ("customers", "customers"), 
                  ("sellers", "sellers"),
                  ("products_clean", "products_clean"), 
                  ("geo_agg", "geo_agg")]:
    try:
        df = eval(var)
        print(f"  {name}: {df.shape} — IN MEMORY")
    except NameError:
        print(f"  {name}: NOT IN MEMORY")

  orders: (99441, 15) — IN MEMORY
  order_items: (112650, 7) — IN MEMORY
  payments_agg: (99437, 5) — IN MEMORY
  reviews_clean: (98410, 8) — IN MEMORY
  customers: (99441, 5) — IN MEMORY
  sellers: (3095, 4) — IN MEMORY
  products_clean: (32951, 10) — IN MEMORY
  geo_agg: (19011, 5) — IN MEMORY


In [23]:
from pathlib import Path
import pandas as pd

CLEANED = Path("../data/cleaned")
CLEANED.mkdir(parents=True, exist_ok=True)

orders.to_csv(CLEANED / "orders_clean.csv", index=False)
order_items.to_csv(CLEANED / "order_items_clean.csv", index=False)
payments_agg.to_csv(CLEANED / "payments_clean.csv", index=False)
reviews_clean.to_csv(CLEANED / "reviews_clean.csv", index=False)
customers.to_csv(CLEANED / "customers_clean.csv", index=False)
sellers.to_csv(CLEANED / "sellers_clean.csv", index=False)
products_clean.to_csv(CLEANED / "products_clean.csv", index=False)
geo_agg.to_csv(CLEANED / "geolocation_clean.csv", index=False)

print("All cleaned files exported.")
for f in sorted(CLEANED.iterdir()):
    df = pd.read_csv(f)
    print(f"  {f.name}: {df.shape}")

All cleaned files exported.
  customers_clean.csv: (99441, 5)
  geolocation_clean.csv: (19011, 5)
  order_items_clean.csv: (112650, 7)
  orders_clean.csv: (99441, 15)
  payments_clean.csv: (99437, 5)
  products_clean.csv: (32951, 10)
  reviews_clean.csv: (98410, 8)
  sellers_clean.csv: (3095, 4)
